# Day2 모델링 (4) - Optuna 튜닝과 앙상블

- 피처: 전체(8,577개) 중 **fold 안에서 중요도 상위 1,000개**만 사용(`build_selected_pipeline`). 04에서 상위 1,000개가 전체와 거의 같은 성능임을 확인했다.
- 튜닝 대상: LightGBM, XGBoost. 목적함수는 5-Fold 평균 **ROC-AUC**다. F1은 임계값에 따라 들쭉날쭉해서 탐색 목적함수로 불안정하다. F1은 튜닝 뒤 OOF 확률로 임계값을 정해서 최대화한다.
- Tanimoto 커널 SVM(C)은 짧게 탐색하고, 세 모델의 확률을 평균해 앙상블한다.
- 홀드아웃은 열지 않는다. 같은 CV로 튜닝했으므로 CV 점수는 약간 낙관적이다.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from xgboost import XGBClassifier

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from data import load_all  # noqa: E402
from models import TanimotoSVC  # noqa: E402
from preprocess import build_pipeline, build_selected_pipeline  # noqa: E402

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42
TAB = ROOT / "reports/tables"
CACHE = ROOT / "data/cache"
d = load_all(ROOT, seed=SEED, extra=True)
X_tr, y_tr, GROUPS = d.X_tr, d.y_tr, d.groups
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
THS = np.round(np.arange(0.30, 0.71, 0.01), 2)
ALL = ["cont", "rdkit", "fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]
SVC_GROUPS = ["fp", "maccs", "morgan_cnt", "ap_cnt", "tt_cnt"]
N_TRIALS = 40


def pipe_selected(model):
    return build_selected_pipeline(model, GROUPS, ALL, k=1000, seed=SEED)


# 진행 상황은 data/cache/optuna_progress.log에 한 줄씩 남기고, 시도 결과는 sqlite에 저장해서 중단 후 이어서 할 수 있게 한다
PROGRESS = CACHE / "optuna_progress.log"
DB = f"sqlite:///{CACHE / 'optuna_v2.db'}"


def log_trial(study, trial):
    with open(PROGRESS, "a", encoding="utf-8") as f:
        f.write(f"{study.study_name} 시도 {trial.number + 1}: AUC {trial.value:.4f} (현재 최고 {study.best_value:.4f})\n")


def run_study(name, objective):
    study = optuna.create_study(study_name=name, storage=DB, load_if_exists=True, direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    done = len([t for t in study.trials if t.state.name == "COMPLETE"])
    study.optimize(objective, n_trials=max(0, N_TRIALS - done), callbacks=[log_trial])
    return study

/Users/t2222h/skala/26_data-mini-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. LightGBM 튜닝

In [2]:
def make_lgbm(p):
    return LGBMClassifier(**p, subsample_freq=1, random_state=SEED, n_jobs=3, verbose=-1)


def objective_lgbm(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 300, 1200),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int("num_leaves", 8, 96),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 60),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.1, 0.6),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
    )
    return cross_val_score(pipe_selected(make_lgbm(p)), X_tr, y_tr, cv=CV, scoring="roc_auc", n_jobs=5).mean()


study_lgbm = run_study("lgbm_v2", objective_lgbm)
print(f"LightGBM 최고 CV AUC {study_lgbm.best_value:.4f} (기본 설정 약 0.883) | trial {len(study_lgbm.trials)}")
print(study_lgbm.best_params)

LightGBM 최고 CV AUC 0.8870 (기본 설정 약 0.883) | trial 40
{'n_estimators': 919, 'learning_rate': 0.011946480235979145, 'num_leaves': 78, 'min_child_samples': 15, 'colsample_bytree': 0.5749540392422449, 'subsample': 0.8943645385496045, 'reg_alpha': 0.01668991789098633, 'reg_lambda': 0.02245444338669948}


## 2. XGBoost 튜닝

In [3]:
def make_xgb(p):
    return XGBClassifier(**p, tree_method="hist", n_jobs=3, random_state=SEED, eval_metric="logloss")


def objective_xgb(trial):
    p = dict(
        n_estimators=trial.suggest_int("n_estimators", 300, 1200),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.1, 0.6),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
        gamma=trial.suggest_float("gamma", 0.0, 5.0),
    )
    return cross_val_score(pipe_selected(make_xgb(p)), X_tr, y_tr, cv=CV, scoring="roc_auc", n_jobs=5).mean()


study_xgb = run_study("xgb_v2", objective_xgb)
print(f"XGBoost 최고 CV AUC {study_xgb.best_value:.4f} (기본 설정 약 0.881) | trial {len(study_xgb.trials)}")
print(study_xgb.best_params)

XGBoost 최고 CV AUC 0.8863 (기본 설정 약 0.881) | trial 40
{'n_estimators': 832, 'learning_rate': 0.02188048719622109, 'max_depth': 9, 'min_child_weight': 2, 'subsample': 0.6711175100841916, 'colsample_bytree': 0.44383759668713557, 'reg_alpha': 0.2920714384846498, 'reg_lambda': 0.015080016284530104, 'gamma': 0.1190476597455401}


## 3. Tanimoto SVM의 C 선택 (짧게)

In [4]:
svc_auc = {}
for C in (3, 10, 30):
    svc_auc[C] = cross_val_score(build_pipeline(TanimotoSVC(C=C), GROUPS, SVC_GROUPS), X_tr, y_tr, cv=CV,
                                 scoring="roc_auc", n_jobs=5).mean()
    print(f"TanimotoSVC C={C}: CV AUC {svc_auc[C]:.4f}")
best_C = max(svc_auc, key=svc_auc.get)
print("선택 C =", best_C)

best = {"lightgbm": study_lgbm.best_params, "xgboost": study_xgb.best_params, "svc_C": best_C,
        "cv_auc": {"lightgbm": study_lgbm.best_value, "xgboost": study_xgb.best_value, "svc": svc_auc[best_C]}}
(TAB / "best_params.json").write_text(json.dumps(best, indent=2, ensure_ascii=False), encoding="utf-8")

TanimotoSVC C=3: CV AUC 0.8733


TanimotoSVC C=10: CV AUC 0.8743


TanimotoSVC C=30: CV AUC 0.8692
선택 C = 10


759

## 4. 튜닝된 모델의 OOF 확률, 앙상블, 임계값

각 모델의 5-Fold 검증 예측 확률(OOF)을 모아 단일 모델과 평균 앙상블의 AUC와 F1을 비교하고 F1이 최대인 임계값을 찾는다.

In [5]:
OOF = {}
OOF["LightGBM(tuned)"] = cross_val_predict(pipe_selected(make_lgbm(study_lgbm.best_params)), X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
OOF["XGBoost(tuned)"] = cross_val_predict(pipe_selected(make_xgb(study_xgb.best_params)), X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
OOF["TanimotoSVC"] = cross_val_predict(build_pipeline(TanimotoSVC(C=best_C), GROUPS, SVC_GROUPS), X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=5)[:, 1]
pd.to_pickle(OOF, CACHE / "oof_05.pkl")


def summarize(name, proba):
    f1s = np.array([f1_score(y_tr, (proba >= t).astype(int)) for t in THS])
    return {"구성": name, "AUC": roc_auc_score(y_tr, proba), "F1@0.5": f1_score(y_tr, (proba >= 0.5).astype(int)),
            "최대 F1": f1s.max(), "최적 임계값": THS[f1s.argmax()]}


rows = [summarize(k, v) for k, v in OOF.items()]
rows.append(summarize("LGBM+XGB 평균", (OOF["LightGBM(tuned)"] + OOF["XGBoost(tuned)"]) / 2))
rows.append(summarize("LGBM+XGB+SVC 평균", np.mean(list(OOF.values()), axis=0)))
rows.append(summarize("LGBM*0.4+XGB*0.4+SVC*0.2", 0.4 * OOF["LightGBM(tuned)"] + 0.4 * OOF["XGBoost(tuned)"] + 0.2 * OOF["TanimotoSVC"]))
res = pd.DataFrame(rows)
res.round(4).to_csv(TAB / "modeling_tuned_ensembles.csv", index=False)
res.round(4)

,구성,AUC,F1@0.5,최대 F1,최적 임계값
0,LightGBM(tuned),0.8868,0.8295,0.8311,0.40
1,XGBoost(tuned),0.8861,0.8277,0.8329,0.33
2,TanimotoSVC,0.8739,0.8201,0.8234,0.45
3,LGBM+XGB 평균,0.8873,0.8266,0.8324,0.45
4,LGBM+XGB+SVC 평균,0.8889,0.8320,0.8334,0.38
5,LGBM*0.4+XGB*0.4+SVC*0.2,0.8889,0.8294,0.8336,0.35
